# SqueezeCtrl — Raw Instrument Test Bench

Standalone, GUI-free tests for every instrument interaction `SqueezeCtrl` relies on
(`src/instrument.py` + `src/config.py`). Run this against the real pressure
controller to verify each SCPI command actually does what the app assumes,
without going through the PyQt UI.

**How to use it:** run top to bottom. Section 1 discovers VISA resources so you
can fill in `VISA_ADDRESS` in Section 2. Section 3 sends each raw SCPI command
one at a time and prints the raw response — compare it against the instrument's
manual. Section 4 re-runs the same checks through the app's own `PressureInstrument`
wrapper, to confirm the wrapper matches the raw behavior.

**⚠️ Safety:** the "set control / set pressure / set rate" cells actually energize
the instrument's output and can move a real actuator. Only run them when it's
safe to do so on your bench, and start with a small, known-safe setpoint.


## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Make `src` importable whether Jupyter was launched from the repo root or from notebooks/
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break

import pyvisa

from src import config as cfg

print("Commands under test (from src/config.py):")
for name in sorted(vars(cfg)):
    if name.startswith("CMD_"):
        print(f"  {name} = {getattr(cfg, name)!r}")


## 1. Discover resources

Mirrors `PressureInstrument.list_candidate_resources()`: lists every VISA
resource, then narrows it down to USB / TCPIP transports.


In [ ]:
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("All VISA resources:", resources)

candidates = [r for r in resources if r.upper().startswith("USB") or r.upper().startswith("TCPIP")]
print("USB / TCPIP candidates:", candidates)


### 1b. Recognize candidates (raw equivalent of `find_recognized_resources`)

Probes each candidate with the mode query and keeps the ones that answer like
a supported pressure controller (response contains `MEAS` or `CONT`).


In [ ]:
recognized = []
for addr in candidates:
    try:
        probe = rm.open_resource(addr)
        probe.timeout = cfg.INSTRUMENT_TIMEOUT_MS
        raw = probe.query(cfg.CMD_QUERY_MODE).strip().upper()
        probe.close()
        print(f"{addr}: {raw!r}")
        if "MEAS" in raw or "CONT" in raw:
            recognized.append(addr)
    except pyvisa.VisaIOError as exc:
        print(f"{addr}: probe failed ({exc})")

print("Recognized as pressure controllers:", recognized)


## 2. Connect

Set `VISA_ADDRESS` to one of the resource strings found above, then run this
cell to open a persistent connection reused by every cell below.


In [ ]:
VISA_ADDRESS = ""  # e.g. "TCPIP0::192.168.1.50::INSTR" or "USB0::0x1234::0x5678::SN123::INSTR"

assert VISA_ADDRESS, "Set VISA_ADDRESS above to a resource string from Section 1 before continuing."

resource = rm.open_resource(VISA_ADDRESS)
resource.timeout = cfg.INSTRUMENT_TIMEOUT_MS
print(f"Connected to {VISA_ADDRESS}")


## 3. Raw SCPI probes

Each cell sends exactly the command string from `src/config.py` and prints the
raw response, so you can check it against the instrument's SCPI reference and
edit `config.py` directly if a command needs fixing.


### 3.1 Mode query — `CMD_QUERY_MODE`

In [ ]:
raw = resource.query(cfg.CMD_QUERY_MODE)
print(f"{cfg.CMD_QUERY_MODE!r} -> {raw!r}")


### 3.2 Read pressure — `CMD_READ_PRESSURE`

In [ ]:
raw = resource.query(cfg.CMD_READ_PRESSURE)
print(f"{cfg.CMD_READ_PRESSURE!r} -> {raw!r}")
try:
    print("parsed as float:", float(raw.strip()))
except ValueError as exc:
    print("!! could not parse as float:", exc)


### 3.3 Read slew rate — `CMD_READ_RATE`

In [ ]:
raw = resource.query(cfg.CMD_READ_RATE)
print(f"{cfg.CMD_READ_RATE!r} -> {raw!r}")
try:
    print("parsed as float:", float(raw.strip()))
except ValueError as exc:
    print("!! could not parse as float:", exc)


### 3.4 Read source pressure — `CMD_READ_SOURCE_PRESSURE`

⚠️ This one looks suspect: it's a query (`?`) but has a literal `1` embedded
before the `?`, which is unusual SCPI syntax (queries don't normally carry a
parameter). If this errors or returns something unexpected, try the plain
query form commented out below and update `config.py` if it works better.


In [ ]:
raw = resource.query(cfg.CMD_READ_SOURCE_PRESSURE)
print(f"{cfg.CMD_READ_SOURCE_PRESSURE!r} -> {raw!r}")
try:
    print("parsed as float:", float(raw.strip()))
except ValueError as exc:
    print("!! could not parse as float:", exc)

# If the above fails, try the plain query instead:
# raw = resource.query(":SOURce:PRESsure:COMPensate?")
# print(raw)


### 3.5 Switch to CONTROL mode — `CMD_SET_SOURCE` + `CMD_SET_OUTPUT`

⚠️ Energizes the output. Pick a small, safe setpoint for your bench.


In [ ]:
TEST_SETPOINT_BAR = 0.5  # keep small/safe for a bench test

resource.write(f"{cfg.CMD_SET_SOURCE} {TEST_SETPOINT_BAR}")
resource.write(f"{cfg.CMD_SET_OUTPUT} 1")
print(f"Sent: '{cfg.CMD_SET_SOURCE} {TEST_SETPOINT_BAR}' then '{cfg.CMD_SET_OUTPUT} 1'")

# Confirm the mode actually switched
print("Mode after switch:", resource.query(cfg.CMD_QUERY_MODE).strip())


### 3.6 Switch to MEASURE mode — `CMD_SET_OUTPUT`

In [ ]:
resource.write(f"{cfg.CMD_SET_OUTPUT} 0")
print(f"Sent: '{cfg.CMD_SET_OUTPUT} 0'")
print("Mode after switch:", resource.query(cfg.CMD_QUERY_MODE).strip())


### 3.7 Set pressure setpoint — `CMD_SET_PRESSURE`

⚠️ This command lacks the `:` prefix that every other command in `config.py`
uses (`"SET:PRES"` vs. e.g. `":SOUR"`). If the instrument doesn't react (no
error, but `read_pressure`/the setpoint readback doesn't move), that
inconsistency is a likely culprit — check the manual for the exact syntax.


In [ ]:
resource.write(f"{cfg.CMD_SET_PRESSURE} {TEST_SETPOINT_BAR}")
print(f"Sent: '{cfg.CMD_SET_PRESSURE} {TEST_SETPOINT_BAR}'")


### 3.8 Set slew rate — `CMD_SET_RATE`

Same leading-`:` inconsistency as 3.7 — flag it if it doesn't take effect.


In [ ]:
TEST_RATE = 1.0

resource.write(f"{cfg.CMD_SET_RATE} {TEST_RATE}")
print(f"Sent: '{cfg.CMD_SET_RATE} {TEST_RATE}'")


## 4. Wrapper class smoke test

Re-run the same checks through `PressureInstrument` (what the GUI actually
uses) to confirm it behaves the same as the raw calls above.


In [ ]:
from src.instrument import ControlMode, InstrumentError, PressureInstrument

instrument = PressureInstrument()
instrument.connect(VISA_ADDRESS)
print("is_connected:", instrument.is_connected)


In [ ]:
print("mode:", instrument.read_mode())
print("pressure:", instrument.read_pressure())
print("rate:", instrument.read_rate())
print("source pressure:", instrument.read_source_pressure())


⚠️ Energizes the output, same as 3.5/3.7 above.

In [ ]:
instrument.set_control(TEST_SETPOINT_BAR)
print("mode after set_control:", instrument.read_mode())


In [ ]:
instrument.set_rate(TEST_RATE)
print("rate after set_rate:", instrument.read_rate())


In [ ]:
instrument.set_measure()
print("mode after set_measure:", instrument.read_mode())


## 5. Cleanup

In [ ]:
resource.close()
instrument.disconnect()
print("Closed connections.")
